# Cellpose finetuning
A notebook to evaluate finetuning performance of various cellpose fine-tuned models

In [ ]:
import os
import json
from pathlib import Path
import re
from sphero_vem.io import imread_labels_downscaled, imread_downscaled
from dotenv import load_dotenv
import numpy as np
import tifffile
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import ListedColormap
import seaborn as sns
from skimage import measure


load_dotenv("../.env")
DATA_ROOT = Path(os.environ.get("DATA_ROOT"))
dir_gt = DATA_ROOT / "processed/labeled/Au_01-vol_01/labeled-01"
dir_pred = DATA_ROOT / "processed/segmented/finetuning"
dir_pretrained = DATA_ROOT / "processed/segmented/pretrained"
dir_save = DATA_ROOT / "figures/segmentation/finetuning"
dir_save.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "Helvetica"


# Common variables
seg_targets = ["cells", "nuclei"]

In [ ]:
def plot_diff(
    ax,
    ground_truth,
    predictions,
    cmap="coolwarm",
    colorbar_width="5%",
    colorbar_pad=0.1,
):
    """Plot difference between between ground truth and predictions"""

    # Map false positive to -1 and false negatives to +1
    diff_mask = (ground_truth > 0) * 1 - (predictions > 0) * 1

    # Create discrete colormap
    colormap = mpl.colormaps[cmap]
    colors = [colormap(0.05), colormap(0.5), colormap(0.95)]
    discrete_cmap = ListedColormap(colors)

    im = ax.imshow(diff_mask, cmap=discrete_cmap, vmin=-1.5, vmax=1.5)

    # Create colorbar axis with controlled size
    cax = inset_axes(
        ax,
        width=colorbar_width,
        height="100%",
        loc="center left",
        bbox_to_anchor=(1 + colorbar_pad, 0.0, 1, 1),
        bbox_transform=ax.transAxes,
        borderpad=0,
    )

    boundaries = [-1.5, -0.5, 0.5, 1.5]
    cbar = plt.colorbar(
        im, cax=cax, boundaries=boundaries, ticks=[-1, 0, 1], extend="neither"
    )
    cbar.set_ticklabels(["FP", "TP", "FN"])
    for spine in cbar.ax.spines.values():
        spine.set_visible(False)

    return cax


def find_contours(labels: np.ndarray) -> list[np.ndarray]:
    """Find external contours for each predicted label"""
    contours = []
    label_vals = np.unique(labels)[1:]
    for val in label_vals:
        contours_val = measure.find_contours(labels == val)
        contours.append(sorted(contours_val, reverse=True, key=len)[0])
    return contours


def load_results(dir_model: Path) -> pd.DataFrame:
    """Load results from a model directory and return as DataFrame.

    Parameters
    ----------
    dir_model : Path
        Path to the model directory containing results and manifest. If there's
        no manifest, n_epochs and downscaling_training are set to pd.NA

    Returns
    -------
    pd.DataFrame
        DataFrame containing evaluation results with model metadata including
        model_name, downscaling factors, training epochs, and file paths
    """
    results = []
    try:
        with open(dir_model / "training_manifest.json", "r") as f:
            manifest = json.load(f)
    except FileNotFoundError:
        # Assign empty dictionary if not found. This will lead to pd.NA assignments
        manifest = {}
    model_type = dir_model.parent.name.replace("ing", "ed")
    n_epochs = (
        manifest.get("n_epochs", pd.NA) if model_type == "finetuned" else "pretrained"
    )
    factor_train = (
        int(manifest["preprocessing_steps"][0]["factor"])
        if model_type == "finetuned"
        else "pretrained"
    )
    for results_path in dir_model.rglob("*-ap.csv"):
        ds_match = re.search(r"downscaled-(\d+)", results_path.parent.name)
        seg_target = re.search(r"-(\w+)-ap.csv", results_path.name).group(1)
        factor = int(ds_match.groups()[0]) if ds_match else factor_train
        masks_path = results_path.parent / results_path.name.replace("-ap.csv", ".tif")
        gt_path = dir_gt / "labels" / masks_path.name
        results_df = pd.read_csv(results_path)
        results_df = results_df.assign(
            **{
                "model_name": dir_model.name,
                "downscaling_training": factor_train,
                "n_epochs": n_epochs,
                "downscaling_factor": factor,
                "ground_truth_path": gt_path,
                "prediction_path": masks_path,
                "model_type": model_type,
                "segmentation_target": seg_target,
            }
        )
        results.append(results_df)
    return pd.concat(results)

In [ ]:
# Load results and pretreat data
results = []
dirs = list(dir_pred.glob("*/")) + list(dir_pretrained.glob("*/"))
for dir_model in dirs:
    results.append(load_results(dir_model))

data = pd.concat(results)
data.duplicated(
    subset=["model_name", "downscaling_factor", "ground_truth_path", "iou_thresholds"]
).sum()

# Manually assign missing values
data.loc[data["model_name"] == "cellposeSAM-cells-ds16-20250527_125402", "n_epochs"] = (
    200
)
data = data.rename(columns={"iou_thresholds": "iou_threshold"})

# Find best model at each downscaling based on maximum average precision
best_models = (
    data.query("downscaling_factor==downscaling_training")
    .groupby(
        [
            "model_name",
            "downscaling_factor",
            "downscaling_training",
            "segmentation_target",
        ],
        as_index=False,
    )["average_precision"]
    .aggregate("mean")
    .sort_values("average_precision", ascending=False)
    .drop_duplicates(["downscaling_factor", "segmentation_target"])
    .loc[:, "model_name"]
    .tolist()
)

In [ ]:
# Performance at training resolutions
for seg_target in ["cells", "nuclei"]:
    data_plot = data.query(
        f"downscaling_factor==downscaling_training & model_type=='finetuned' & segmentation_target=='{seg_target}'"
    )
    g = sns.relplot(
        data_plot,
        x="iou_threshold",
        y="average_precision",
        style="n_epochs",
        hue="downscaling_factor",
        kind="line",
        errorbar=None,
    )
    g.ax.set(
        xlim=(0.05, 0.95),
        ylim=(0, 1),
        xlabel="IoU threshold",
        ylabel="Average precision",
    )
    g.savefig(dir_save / f"cellposeSAM-{seg_target}-finetuning-native-all.png", dpi=300)

In [ ]:
# Performance at native model resolution
factors = [5, 8, 10, 16, 20]
seg_targets = ["cells", "nuclei"]

for seg_target in seg_targets:
    for factor in factors:
        data_plot = data.query(
            f"downscaling_training==[{factor}, 'pretrained'] & downscaling_factor=={factor}"
            + f"& segmentation_target=='{seg_target}'"
        )
        hue_order = (
            data_plot["n_epochs"].unique()
            if factor != 10
            else [351, 501, 1001, "pretrained"]
        )
        g = sns.relplot(
            data_plot,
            x="iou_threshold",
            y="average_precision",
            hue="n_epochs",
            style="downscaling_training",
            kind="line",
            hue_order=hue_order,
            errorbar=None,
        )
        g.ax.set(
            xlim=(0.05, 0.95),
            ylim=(0, 1),
            xlabel="IoU threshold",
            ylabel="Average precision",
            title=f"Precision of models trained at downscaling factor {factor}",
        )
        g.savefig(
            dir_save / f"cellposeSAM-{seg_target}-finetuning-AP-ds{factor}-native.png",
            dpi=300,
        )

In [ ]:
models = best_models + ["cellposeSAM"]
for seg_target in seg_targets:
    data_plot = data.query(
        "(downscaling_factor==downscaling_training | model_type=='pretrained') & downscaling_factor==[5, 8, 10, 16, 20]"
        + "& model_name==@models & segmentation_target==@seg_target"
    )
    g = sns.catplot(
        data_plot,
        x="downscaling_factor",
        y="average_precision",
        kind="bar",
        errorbar=None,
        hue="model_type",
        hue_order=["pretrained", "finetuned"],
    )
    g.ax.set(
        ylim=(0, 1),
        xlabel="Downscaling factor",
        ylabel="Mean average precision across all thresholds",
        title="Best models at native training resolution",
    )
    g.refline(y=0.85)
    g.ax.yaxis.set_minor_locator(plt.MultipleLocator(0.1))
    g.ax.tick_params(axis="y", which="minor", left=True)

    g.savefig(
        dir_save / f"cellposeSAM-{seg_target}-finetuning-meanAP-native.png", dpi=300
    )

In [ ]:
models = best_models + ["cellposeSAM"]
for seg_target in seg_targets:
    for factor in [1, 5, 10]:
        data_plot = data.query(
            "downscaling_factor==@factor & model_name==@models"
            + "& segmentation_target==@seg_target"
        )

        g = sns.relplot(
            data_plot,
            x="iou_threshold",
            y="average_precision",
            hue="downscaling_training",
            style="model_type",
            kind="line",
            hue_order=[5, 8, 10, 16, 20, "pretrained"],
            errorbar=None,
        )
        g.ax.set(
            xlim=(0.05, 0.95),
            ylim=(0, 1),
            xlabel="IoU threshold",
            ylabel="Average precision",
            title=f"Precision of {seg_target} models evaluated at downscaling factor {factor}",
        )
        g.savefig(
            dir_save
            / f"cellposeSAM-{seg_target}-finetuning-AP-upscaled-ds{factor}.png",
            dpi=300,
        )

In [ ]:
# Plot visual evaluations for best cell models
model_name = "cellposeSAM-cells-ds10-20250605_111901"

images_data = data.query(
    "model_name==@model_name & downscaling_factor==downscaling_training"
)
images_data = images_data.drop_duplicates("ground_truth_path")

for _, row in images_data.iterrows():
    image_path = row["ground_truth_path"].parents[1] / row[
        "ground_truth_path"
    ].name.replace("-cells", "")
    image = imread_downscaled(image_path, row["downscaling_factor"])
    ground_truth = imread_labels_downscaled(
        row["ground_truth_path"], row["downscaling_factor"]
    )
    predictions = tifffile.imread(row["prediction_path"])

    contours = [find_contours(ground_truth), find_contours(predictions)]
    titles = ["Ground truth", "Predictions", "Difference (ground truth - predictions)"]
    diff_mask = (ground_truth > 0) * 1 - (predictions > 0) * 1

    fig, ax = plt.subplots(1, 3, dpi=250, layout="constrained", figsize=(10, 5))

    for i, contours_image in enumerate(contours):
        ax[i].imshow(image, cmap="gray", vmin=0, vmax=200)
        for contour in contours_image:
            ax[i].plot(contour[:, 1], contour[:, 0], color="y", linewidth=1, ls="--")

    cax = plot_diff(ax[2], ground_truth, predictions, colorbar_pad=0.02)

    for i, axis in enumerate(ax):
        axis.set_axis_off()
        axis.set(title=titles[i])

    fig.savefig(
        dir_save / f"{model_name}-{image_path.stem}-comparison.png",
        bbox_extra_artists=[cax],
        bbox_inches="tight",
        dpi=300,
    )

In [ ]:
# Plot visual evaluations for best nuclei models
model_name = "cellposeSAM-nuclei-ds10-20250610_094212"

images_data = data.query(
    "model_name==@model_name & downscaling_factor==downscaling_training"
)
images_data = images_data.drop_duplicates("ground_truth_path")

for _, row in images_data.iterrows():
    image_path = row["ground_truth_path"].parents[1] / row[
        "ground_truth_path"
    ].name.replace(f"-{row['segmentation_target']}", "")
    image = imread_downscaled(image_path, row["downscaling_factor"])
    ground_truth = imread_labels_downscaled(
        row["ground_truth_path"], row["downscaling_factor"]
    )
    predictions = tifffile.imread(row["prediction_path"])

    contours = [find_contours(ground_truth), find_contours(predictions)]
    titles = ["Ground truth", "Predictions", "Difference (ground truth - predictions)"]
    diff_mask = (ground_truth > 0) * 1 - (predictions > 0) * 1

    fig, ax = plt.subplots(1, 3, dpi=250, layout="constrained", figsize=(10, 5))

    for i, contours_image in enumerate(contours):
        ax[i].imshow(image, cmap="gray", vmin=0, vmax=200)
        for contour in contours_image:
            ax[i].plot(contour[:, 1], contour[:, 0], color="y", linewidth=1, ls="--")

    cax = plot_diff(ax[2], ground_truth, predictions, colorbar_pad=0.02)

    for i, axis in enumerate(ax):
        axis.set_axis_off()
        axis.set(title=titles[i])

    fig.savefig(
        dir_save / f"{model_name}-{image_path.stem}-comparison.png",
        bbox_extra_artists=[cax],
        bbox_inches="tight",
        dpi=300,
    )